# 第4周 · 周四：Hive与大数据处理——当数据量大到Excel打不开

> 难度：⭐⭐⭐⭐ | 定位：理解Hive/Hadoop的核心原理，掌握大数据环境下的SQL编写和优化策略
> 前置依赖：周一至周三的数仓流转与分层知识 | 后续用途：在大数据平台上高效完成分析任务
> 面试地位：大厂数据分析面试必考，约占数据工程类问题的30%

---

## 📘 区块1：核心概念与第一性原理

### 1.1 为什么需要Hive——从第一性原理推导

在单机MySQL上，处理1000万行数据没有压力。但如果你面对的是100亿行——MySQL会直接挂掉。不是因为MySQL不够好，而是因为单机数据库的设计目标就不是"海量数据分析"。

Hive的设计哲学很简单也很深刻：**把SQL翻译成MapReduce/Spark任务，分发到几十甚至几百台机器上并行计算，最后汇总结果。**

```
你写的SQL
  → Hive解析器：把SQL翻译成执行计划
    → 优化器：选择最优执行路径（MapReduce/Tez/Spark）
      → 分发到100台机器
        → 每台处理1/100的数据
          → 各节点的中间结果Shuffle到Reduce节点
            → 汇总输出
```

这就像：你让100个人同时抄一本书，每人抄10页，最后装订成册。1个人抄1000页需要一周，100个人只需要20分钟。分散处理→并行计算→汇总结果——这是大数据处理的底层逻辑。

但代价是什么？
- **启动慢**：分配100个人需要协调时间——MapReduce任务启动就需要几十秒，哪怕你只查1行数据
- **中间开销大**：每个人抄完还要汇总——Shuffle（中间结果重分布）可能占了50%以上的执行时间
- **不支持实时更新**：你不能让100个人同时在修改的稿子上做笔记——Hive不支持行级UPDATE/DELETE

### 1.2 Hive vs MySQL——不是"谁更好"，而是"完全不同的物种"

很多人一上来就问"Hive比MySQL快吗"——这个问题本身就是错的。它们是为完全不同场景设计的：

| 维度 | MySQL | Hive |
|------|-------|------|
| 设计目标 | 在线事务处理（OLTP） | 离线分析处理（OLAP） |
| 数据量级 | GB级 | TB到PB级 |
| 查询延迟 | 毫秒到秒 | 分钟到小时 |
| 并发模式 | 高并发小查询 | 低并发大查询 |
| 数据更新 | 支持行级UPDATE/DELETE | 不支持（或有限支持） |
| 事务支持 | 完整ACID | 有限或不支持 |
| 索引 | 丰富的索引 | 没有传统索引（靠分区） |
| 存储格式 | 行式存储（InnoDB） | 列式存储（ORC/Parquet） |
| 适用场景 | 在线业务系统 | 离线数据分析 |

这里有一个重要的比喻：
- MySQL像一辆跑车：起步快、加速猛、机动灵活——适合频繁的小任务
- Hive像一列货运火车：启动慢、加速慢，但一次能拉几千吨——适合一次性处理海量数据

你不会用跑车去运铁矿，也不会用货运火车去送外卖。同理——你不会用Hive做实时查询，也不应该用MySQL分析100亿行数据。

### 1.3 Hive的核心约束——理解了就不再踩坑

**约束1：不支持行级UPDATE/DELETE（传统Hive）**
- Hive是为"一次写入、多次读取"设计的
- 数据以文件形式存在HDFS上，文件是不可变的
- 如果要"修改"数据，实际上是重写整个文件
- 这决定了Hive不适合频繁修改的数据（如用户维表每日更新）

**约束2：高查询延迟**
- 即使最简单的`SELECT COUNT(*)`，启动MapReduce就需要几十秒
- 适合T+1的批处理，不适合实时交互式查询
- 如果业务需要秒级响应，应该选ClickHouse/StarRocks/Doris，不是Hive

**约束3：SQL不是100%兼容MySQL**
- HiveQL是SQL的一个子集+扩展
- 某些MySQL函数在Hive中名称或参数不同（如DATE_FORMAT → date_format）
- 某些语法在Hive中不支持（如非等值JOIN的某些写法）

**约束4：不支持事务（传统Hive）**
- 没有BEGIN/COMMIT/ROLLBACK
- 不能保证ACID
- 这意味着"一段SQL中的某一句失败了"不会回滚已经执行的部分

### 1.4 Hive的架构组件

理解Hive的架构组件，你才能在出问题时知道"该找谁"：

```
┌──────────────────────────────────────────┐
│              Hive CLI / JDBC              │  ← 你写SQL的地方
└────────────────┬─────────────────────────┘
                 │
┌────────────────▼─────────────────────────┐
│            HiveServer2                    │  ← 接收SQL请求
└────────────────┬─────────────────────────┘
                 │
┌────────────────▼─────────────────────────┐
│            Driver / Compiler              │  ← 解析+编译SQL
│    SQL → AST → Query Block → DAG         │
└────────────────┬─────────────────────────┘
                 │
┌────────────────▼─────────────────────────┐
│            Optimizer                      │  ← 优化执行计划
│    选择MapReduce/Tez/Spark引擎            │
└────────────────┬─────────────────────────┘
                 │
┌────────────────▼─────────────────────────┐
│         Execution Engine                  │  ← 真正干活的人
│    YARN分配资源 → Container跑Task         │
└────────────────┬─────────────────────────┘
                 │
┌────────────────▼─────────────────────────┐
│              HDFS (存储)                   │  ← 数据存放的地方
│    文件分Block(128MB) 分散在各个节点        │
└──────────────────────────────────────────┘
```

**你作为分析师需要知道的**：
- HiveServer2如果挂了 → SQL全部不能跑（找平台运维）
- YARN资源不够 → SQL排队等待（找平台运维加资源）
- HDFS某个DataNode挂了 → 数据自动从副本恢复（不需要你管）
- 你的SQL慢 → 可能是数据量大、倾斜、或SQL写得不好（你自己可以优化）

### 1.5 面试高频考点

**面试官常问**：
- "Hive和MySQL的核心区别是什么？"
- "为什么Hive的查询延迟高？"
- "分区是什么？为什么重要？"
- "Hive不支持行级UPDATE——那数据怎么更新？"
- "SELECT * 在Hive上为什么不好？"

**回答要点**：
- MySQL是OLTP（单机、毫秒级、支持事务），Hive是OLAP（分布式、分钟级、不支持事务）
- 延迟高因为MapReduce启动需要时间、中间Shuffle开销大
- 分区=按字段分目录存数据，查询时只读相关目录（分区裁剪）
- 数据更新=重写整个分区文件，或使用支持ACID的ORC表
- SELECT * 意味着扫描所有列——列式存储下影响更大，浪费IO

## 🧩 区块2：知识点详细拆解

### 2.1 分区——Hive最重要的优化概念

**为什么重要**：分区是Hive性能优化的"第一性原理"——减少不必要的IO。没有分区的Hive表就像一本没有目录的书。

**分区是什么**：把数据按某个字段（通常是日期）分目录存储在HDFS上。

```
没有分区：
  /warehouse/orders/                    ← 100亿行全在一个目录
    part-00000
    part-00001
    ...

有分区（按dt分区）：
  /warehouse/orders/
    dt=2026-01-01/                      ← 每天一个目录
      part-00000
    dt=2026-01-02/
      part-00000
    ...
```

**分区裁剪——最重要的优化**：
```sql
-- 没有分区裁剪：全表扫描100亿行
SELECT COUNT(*) FROM orders WHERE dt = '2026-01-01';
-- 实际效果：dt是分区字段，Hive只读dt=2026-01-01这一个目录！
-- 100亿行 → 只扫5000万行
```

**常见错误**：
```sql
-- ✗ 错误：用LIKE不会触发分区裁剪
SELECT * FROM orders WHERE dt LIKE '2026-01%';
-- 后果：全表扫描！30分钟跑不完

-- ✓ 正确：用范围条件
SELECT * FROM orders WHERE dt >= '2026-01-01' AND dt <= '2026-01-31';
-- 后果：只扫1月的31个分区，1分钟出结果
```

**分区字段的选择**：
| 字段 | 是否适合做分区 | 原因 |
|------|--------------|------|
| 日期(dt) | ✓ 最适合 | 绝大多数查询按日期过滤 |
| 小时 | ✓ 适合 | 当单天数据量也很大时 |
| 城市 | △ 谨慎 | 分区数=城市数，如果只有10个城市可以 |
| user_id | ✗ 不适合 | 分区数=用户数，几百万个分区会炸掉元数据 |
| 订单状态 | ✗ 不适合 | 值太少了（通常就几个），分区不够细 |

### 2.2 列式存储——ORC vs Parquet

**为什么重要**：列式存储是Hive比MySQL（行式存储）适合分析的核心原因之一。

**行式存储 vs 列式存储**：
```
行式存储（MySQL InnoDB）：
  [行1: id│name│city│age│amount]
  [行2: id│name│city│age│amount]
  → SELECT city, SUM(amount) ... 需要读所有行的所有列

列式存储（Hive ORC/Parquet）：
  [列id: 1│2│3│4│5│...]
  [列name: 张三│李四│王五│...]
  [列city: 北京│上海│广州│...]
  [列amount: 100│200│150│...]
  → SELECT city, SUM(amount) ... 只需要读city列和amount列！
```

**为什么SELECT * 在Hive上特别不好**：
- 列式存储下，SELECT * 意味着读所有列的所有数据
- 如果你的表有50列但分析只需要3列——SELECT * 浪费了47列的IO
- 100亿行 × 47列的无用IO = 巨大的浪费

**ORC vs Parquet**：
| 特性 | ORC | Parquet |
|------|-----|---------|
| 压缩比 | 更高 | 高 |
| 查询性能 | Hive原生支持最好 | Spark更喜欢 |
| 嵌套结构 | 支持一般 | 支持更好 |
| 适用场景 | Hive数仓 | 跨平台（Spark/Hive/Presto） |

### 2.3 MAPJOIN——小表驱动大表的艺术

**为什么重要**：MAPJOIN可以完全避免Shuffle，是大表JOIN小表的终极优化。

**普通JOIN的执行过程**：
```
Map阶段：读两个表 → Map输出 → Shuffle（按JOIN key重分布到Reduce节点） → Reduce JOIN
Shuffle是最慢的环节——数据需要在网络上传输
```

**MAPJOIN的执行过程**：
```
Map阶段：把小表全部加载到内存 → 大表每一行直接在内存中查找匹配 → 直接输出
没有Shuffle！没有Reduce！
```

**使用MAPJOIN的条件**：
- 小表必须小到能全部加载到一个节点的内存中（通常<100MB或<1000万行）
- Hive中可以设置：`SET hive.auto.convert.join=true;`
- 也可以手动指定：`SELECT /*+ MAPJOIN(b) */ ... FROM big_table a JOIN small_table b ON ...`

**适用场景**：
- 订单事实表（100亿行）JOIN 用户维表（1000万行）
- 日志表（1000亿行）JOIN 品类维表（500行）
- 任何"大表JOIN小表"的场景都应该用MAPJOIN

### 2.4 数据仓库文件格式选择

| 格式 | 压缩比 | 查询速度 | 可分割 | 适用场景 |
|------|--------|---------|--------|----------|
| TEXTFILE | 无 | 慢 | ✓ | 原始数据暂存（ODS） |
| ORC | 高 | 快 | ✓ | DWD/DWS层（推荐） |
| Parquet | 高 | 快 | ✓ | 跨平台场景 |
| Avro | 中 | 中 | ✓ | 需要Schema演进的场景 |
| SequenceFile | 中 | 中 | ✓ | 中间数据 |

**推荐策略**：ODS层用TEXTFILE（保持原始）→ DWD/DWS/ADS层用ORC（压缩+查询性能）

### 2.5 Hive SQL优化清单

| 优化项 | 做法 | 效果 |
|--------|------|------|
| 分区裁剪 | WHERE里带上分区字段，用=或范围，不用LIKE | 从全表扫描→只读相关分区 |
| 列裁剪 | 只SELECT需要的列，不用SELECT * | 减少列式存储的IO |
| 先过滤再JOIN | WHERE条件尽可能提前 | 减少JOIN处理的数据量 |
| MAPJOIN | 小表JOIN大表时用MAPJOIN | 消除Shuffle |
| 避免数据倾斜 | 对倾斜key单独处理（见明日的课程） | 避免单节点拖累全体 |
| 避免COUNT(DISTINCT) | 改用GROUP BY子查询 | 减少去重的单节点压力 |
| 合理设置并行度 | SET mapreduce.job.reduces=N | 平衡每个Reduce处理的数据量 |

### 2.6 Hive vs Spark SQL vs Presto/Trino

Hive不是唯一的大数据SQL引擎：

| 引擎 | 计算模型 | 延迟 | 适用场景 |
|------|---------|------|----------|
| Hive (on MR) | MapReduce磁盘计算 | 分钟到小时 | 大规模ETL |
| Hive (on Tez) | DAG内存+磁盘 | 数十秒到分钟 | ETL+中等交互 |
| Hive (on Spark) | Spark DAG | 数十秒到分钟 | ETL+中等交互 |
| Spark SQL | Spark原生 | 数秒到数十秒 | 交互式+ETL |
| Presto/Trino | 纯内存MPP | 秒级 | 交互式查询 |
| StarRocks/Doris | MPP+向量化 | 毫秒到秒 | 实时看板+交互 |

**选择策略**：ETL用Hive/Spark，交互式查询用Presto/StarRocks，一个公司通常同时使用多种引擎。

### 2.7 HDFS基础——数据到底存在哪

**为什么重要**：了解HDFS的基本概念，你才能理解"分区的本质是分目录"、"列式存储的优势"等概念。

**HDFS的核心理念**：
- 文件被切成Block（通常128MB），每个Block有3份副本分布在不同节点
- 计算向数据移动：数据在哪台机器上，任务就调度到哪台机器上执行
- 这减少了网络传输——计算直接在数据所在的节点上进行

**Block vs 分区**：
- Block是物理存储单元（HDFS层面，你不可控）
- 分区是逻辑组织方式（Hive层面，你可以设计）
- 一个好的分区设计 = 查询时只需读取相关分区的Block

### 2.8 YARN资源管理——为什么你的SQL在排队

YARN负责分配集群的计算资源（CPU和内存）。当你说"我的SQL等了20分钟才开始跑"——原因是YARN在排队。

**关键概念**：
- Container = 一个计算单元（一定量的CPU+内存）
- 你的SQL需要的Container数 = Map数 + Reduce数
- 如果集群总资源不够 → 排队等待
- 如果某个Container挂了 → YARN会重新分配

**你能做的**：
- 减少Reduce数量（合理设置），降低资源需求
- 避免在大促/高峰时段跑大SQL
- 如果经常排队，推动平台运维加资源

## 💻 区块3：动手练习

### 步骤0：准备——理解Hive的执行计划

In [ ]:
# ===== 步骤0：Hive EXPLAIN分析 =====

print("=" * 60)
print("Hive EXPLAIN 输出解读")
print("=" * 60)
print()

print("在Hive中，EXPLAIN能显示SQL的执行计划：")
print()
print("EXPLAIN SELECT city, SUM(amount) AS gmv")
print("FROM orders")
print("WHERE dt = '2026-01-15'")
print("GROUP BY city;")
print()

print("关键输出解读：")
print("  TableScan → 扫描哪个表、哪个分区")
print("    partition: dt=2026-01-15  ← 确认分区裁剪生效！")
print("  Group By Operator → GROUP BY操作")
print("    keys: city")
print("    aggregations: sum(amount)")
print("  Reduce Output Operator → Reduce阶段")
print()

print("【重点看懂三段】")
print("  1. TableScan的partition → 确认分区裁剪是否生效")
print("  2. Map Join → 确认MAPJOIN是否生效")
print("  3. number of mappers/reducers → 确认并行度")


### 步骤1：基础练习——分区裁剪的正确写法

In [ ]:
# ===== 步骤1：分区裁剪的正确和错误写法 =====

print("=" * 60)
print("分区裁剪——一个字符的区别，查询时间差100倍")
print("=" * 60)
print()

examples = [
    ("✓ 范围查询",
     "WHERE dt >= '2026-01-01' AND dt <= '2026-01-31'",
     "触发分区裁剪，只读1月31个分区"),
    ("✓ BETWEEN",
     "WHERE dt BETWEEN '2026-01-01' AND '2026-01-31'",
     "触发分区裁剪，等价于范围查询"),
    ("✓ 等值查询",
     "WHERE dt = '2026-01-15'",
     "触发分区裁剪，只读一个分区"),
    ("✓ IN查询",
     "WHERE dt IN ('2026-01-01', '2026-01-02')",
     "触发分区裁剪，只读两个分区"),
    ("✗ LIKE查询",
     "WHERE dt LIKE '2026-01%'",
     "不触发分区裁剪！全表扫描！"),
    ("✗ 函数包裹",
     "WHERE DATE_FORMAT(dt, '%Y-%m') = '2026-01'",
     "分区字段被函数包裹→不触发裁剪→全表扫描"),
    ("✗ 子查询中",
     "WHERE dt = (SELECT MAX(dt) FROM orders)",
     "子查询结果在运行时才知道→分区裁剪无法静态确定"),
]

for label, sql, note in examples:
    print(f"{label}")
    print(f"  SQL:  {sql}")
    print(f"  效果: {note}")
    print()

print("记住：分区裁剪的黄金法则是'让Hive在编译时就知道要读哪些分区'。")
print("LIKE和函数包裹都会破坏这个确定性。")


### 步骤2：进阶练习——SQL优化对比

In [ ]:
# ===== 步骤2：同一个需求，不同写法的性能差异 =====

print("=" * 60)
print("需求：统计昨天各品类的新用户首单金额")
print("=" * 60)
print()

print("【原始写法（性能差）】")
print("-" * 40)
print("""
SELECT category, AVG(amount)
FROM orders o
JOIN users u ON o.user_id = u.user_id
WHERE o.dt = '2026-01-15'
  AND u.register_date = '2026-01-15'
GROUP BY category;
""")
print()
print("问题分析：")
print("  1. users表没有分区过滤→扫描全量用户表（可能5000万行）")
print("  2. 先JOIN再过滤→JOIN了不需要的数据")
print("  3. orders表有dt分区裁剪（OK）")
print("  预估：扫描2亿行 → 可能跑10-30分钟")
print()

print("【优化写法（性能好）】")
print("-" * 40)
print("""
SELECT /*+ MAPJOIN(u) */
       o.category, AVG(o.amount) AS avg_amount
FROM orders o
JOIN (
    SELECT user_id
    FROM users
    WHERE register_date = '2026-01-15'
      AND dt = '2026-01-15'  -- 如果users表也有分区
) u ON o.user_id = u.user_id
WHERE o.dt = '2026-01-15';
""")
print()
print("优化点：")
print("  1. users表先过滤再JOIN→只取当天注册用户")
print("  2. MAPJOIN把过滤后的小表加载到内存→消除Shuffle")
print("  3. 预估：扫描1000万行 → 可能跑30秒-2分钟")
print()
print("性能提升：10-30分钟 → 30秒-2分钟，提升约10-30倍")


### 步骤3：挑战练习——综合优化

In [ ]:
# ===== 步骤3：综合优化练习 =====

print("=" * 60)
print("挑战：优化以下慢查询（100亿行订单表）")
print("=" * 60)
print()

print("【原始慢查询——跑了40分钟还没出】")
print("""
SELECT 
    u.city,
    p.category,
    COUNT(DISTINCT o.user_id) AS buyers,
    SUM(o.amount) AS gmv
FROM orders o
JOIN users u ON o.user_id = u.user_id
JOIN products p ON o.product_id = p.product_id
WHERE o.dt BETWEEN '2026-01-01' AND '2026-01-31'
  AND o.amount > 0
  AND u.status = 1
  AND p.status = 1
GROUP BY u.city, p.category;
""")
print()

print("优化清单：")
print("  1. COUNT(DISTINCT)→先GROUP BY去重再COUNT")
print("  2. 先过滤users表和products表再JOIN")
print("  3. users和products如果是小表→MAPJOIN")
print("  4. orders表分区过滤已OK")
print()

print("【优化后SQL】")
print("""
WITH
-- 1. 先过滤用户表
active_users AS (
    SELECT user_id, city
    FROM users
    WHERE status = 1
),
-- 2. 先过滤商品表
active_products AS (
    SELECT product_id, category
    FROM products
    WHERE status = 1
),
-- 3. 订单数据先做预聚合
order_stats AS (
    SELECT 
        user_id,
        product_id,
        SUM(amount) AS total_amount,
        COUNT(DISTINCT order_id) AS order_count
    FROM orders
    WHERE dt BETWEEN '2026-01-01' AND '2026-01-31'
      AND amount > 0
    GROUP BY user_id, product_id
)
-- 4. JOIN小表+聚合
SELECT /*+ MAPJOIN(au, ap) */
    au.city,
    ap.category,
    COUNT(DISTINCT os.user_id) AS buyers,
    SUM(os.total_amount) AS gmv
FROM order_stats os
JOIN active_users au ON os.user_id = au.user_id
JOIN active_products ap ON os.product_id = ap.product_id
GROUP BY au.city, ap.category;
""")
print()
print("预估提升：40分钟 → 3-5分钟，提升约10倍")


## 📊 区块4：业务场景应用

### 场景1：分区陷阱——LIKE让查询慢了100倍

新来的分析师写了一个SQL：
```sql
SELECT COUNT(DISTINCT user_id) 
FROM orders 
WHERE dt LIKE '2026-01%';
```
本意是"看2026年1月的活跃用户数"。SQL跑了30分钟还没出结果——订单表100亿行，全表扫描了。

为什么？`WHERE dt LIKE '2026-01%'`不会触发分区裁剪！Hive优化器无法识别LIKE模式与分区的对应关系。

正确写法：
```sql
WHERE dt >= '2026-01-01' AND dt <= '2026-01-31'
-- 或
WHERE dt BETWEEN '2026-01-01' AND '2026-01-31'
```
只改了这一行——查询时间从30分钟降到30秒。

**关键启示**：在大数据环境下，"语法正确"不等于"执行正确"。一个看似无害的LIKE，让你的查询慢了100倍。你要理解SQL在幕后会如何执行。

### 场景2：为什么今天同样的SQL跑得飞快，昨天挂掉了

昨天是工作日，今天是618大促。

你的SQL：
```sql
SELECT user_id, SUM(amount) FROM orders 
WHERE dt = '2026-06-18' GROUP BY user_id;
```

平时每天订单量2000万，正常跑完。618当天订单量2亿——数据量涨了10倍。而且出现了大量'超级用户'（一个user_id下几百单）——数据倾斜也变严重了。

同样的SQL、同样的集群、同样的优化——但数据变了。

**关键启示**：SQL性能不是"静态"的——它取决于数据。今天能跑的SQL，明天数据量涨了10倍就可能跑不动。分析师要有"数据量敏感性"——预估数据增长对查询的影响。

### 场景3：从MySQL到Hive——不是你想象的那样

你之前在MySQL中写的SQL：
```sql
SELECT * FROM orders WHERE order_date = '2026-01-01';
```
在Hive中可能会报错——因为Hive区分大小写，字段名可能是`order_date`也可能是`Order_Date`。

你还可能遇到：
- MySQL的`DATE_FORMAT`在Hive中写法不同
- MySQL的`LIMIT 10 OFFSET 100`在Hive中分页写法不同
- MySQL的`INSERT INTO ... VALUES`在Hive中不支持多行
- MySQL的`NOW()`在Hive中是`CURRENT_TIMESTAMP()`

**关键启示**：不要假设"MySQL能跑的SQL在Hive上也能跑"。每次切换平台，先确认语法兼容性。

### 场景4：报表数仓选型——Hive还是ClickHouse？

产品经理想做一个"实时运营看板"——看昨天的数据要等30秒是绝对不能接受的。

Hive不适合这个场景。你需要选择：
- 如果需要秒级甚至毫秒级响应：ClickHouse / StarRocks / Doris
- 如果需要在Hive生态内做交互式：Presto/Trino on Hive
- 如果只是T+1日报：Hive足够

**选型考量**：
| 需求 | 推荐引擎 |
|------|----------|
| T+1日报、离线分析 | Hive |
| 交互式探索分析 | Presto/Trino |
| 实时看板（秒级） | ClickHouse/StarRocks |
| 大规模ETL | Spark/Hive |
| 流式实时分析 | Flink |

## ✅ 区块5：分级检验清单

### 🟢 初级检验（必须能独立完成的）
- [ ] 能解释Hive和MySQL的核心区别（至少5条）
- [ ] 能说出分区的概念和作用
- [ ] 知道SELECT * 在Hive上为什么不好
- [ ] 能写出触发分区裁剪的WHERE条件
- [ ] 能区分行式存储和列式存储

**答案提示**：
- Hive vs MySQL：数据量(TB/PB vs GB)、延迟(分钟 vs 毫秒)、更新(不支持vs支持)、事务(不支持vs支持)、存储(列式vs行式)
- 分区=按字段分目录存数据，查询时只读相关目录
- SELECT * 会读所有列，列式存储下浪费巨大IO
- 分区条件：用等值(=)、范围(BETWEEN/>=)、IN——不要用LIKE或函数包裹分区字段
- 行式：数据按行连续存储；列式：数据按列分别存储

### 🟡 中级检验（能解释给同事听的原理）
- [ ] 能写出触发分区裁剪的多种写法（=、BETWEEN、IN）和避免的写法（LIKE、函数）
- [ ] 能说出MAPJOIN的原理和适用条件
- [ ] 能分析一个Hive SQL的性能瓶颈
- [ ] 能区分Hive/Spark/Presto的适用场景
- [ ] 知道ORC和Parquet的区别和选择

**答案提示**：
- 分区裁剪：=、>=、BETWEEN、IN可以；LIKE、函数包裹、子查询不可以
- MAPJOIN把小表加载到内存，避免Shuffle——条件是"小表足够小"
- 性能瓶颈看：分区是否裁剪、JOIN是否MAPJOIN、是否有数据倾斜
- Hive=ETL重批处理，Spark SQL=中等交互+ETL，Presto=交互式查询
- ORC在Hive生态更优，Parquet跨平台更好

### 🔴 高级检验（能举一反三的迁移能力）
- [ ] 能根据业务场景选择合适的大数据引擎
- [ ] 能设计合理的分区策略（分区字段、粒度、层级）
- [ ] 能写出在100亿行数据上高效运行的SQL
- [ ] 能评估一个Hive SQL的预估耗时和资源消耗
- [ ] 能从分析师视角推动大数据平台的优化

**答案提示**：
- 选型：离线ETL用Hive/Spark，交互式用Presto，实时用Flink/ClickHouse
- 分区策略：首选日期(dt)作为一级分区，数据量特别大时加小时作为二级分区
- 大数据SQL原则：分区裁剪→列裁剪→先过滤→MAPJOIN→避免COUNT(DISTINCT)→抗倾斜
- 预估耗时：行数×每行处理时间÷并行度，100亿行在1000并行度下≈几分钟到几十分钟

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 周一** | 数据流转 | Hive是数据流转中"数仓存储层"的核心引擎——你写的SQL最终是靠Hive执行的 |
| **← 周二** | 数仓分层 | ODS/DWD/DWS/ADS各层表的存储格式和分区策略不同——分层设计在Hive上落地 |
| **← 周三** | SCD与宽表 | Type 2的JOIN在大数据量下的性能要靠Hive的MAPJOIN和分区来保障 |
| **→ 明日** | 数据倾斜 | Hive分布式架构下的最大敌人——倾斜是Hive SQL最常见的性能杀手 |
| **→ 第5-6周** | SQL分析实战 | 所有SQL例题在大数据量下的运行方式需要Hive知识 |
| **业务连接** | 报表查询慢 | 分析师最常遇到的问题——根因往往在Hive的表设计不合理 |

### 横向连接

**与存储优化的关联**：
- 列式存储+压缩可以节省90%的存储空间
- 分区策略直接影响存储和查询效率

**与成本控制的关联**：
- 一个不合理的全表扫描SQL可能烧掉几百元的计算资源
- 合理的分区+列裁剪可以节省大部分成本

**与数据时效性的关联**：
- Hive是T+1离线处理的代表
- 知道Hive的延迟特征才能正确评估"这个指标能不能实时"

## 📝 区块7：当日小结

> **Hive不是"慢的MySQL"——它是为完全不同的场景（PB级离线分析）设计的分布式SQL引擎。理解Hive的核心概念——分区裁剪（减少IO）、列式存储（按需读列）、MAPJOIN（消除Shuffle）、数据倾斜（分布式计算最大的敌人）——你才能写出在大数据环境下"能跑通且跑得快"的SQL。在大数据环境下，写SQL不只是"语法要正确"，更要"知道这段SQL会被如何执行"。一个看似无害的LIKE，可能让你的查询从30秒变成30分钟。**

**三条金句带回家**：
1. 分区裁剪是Hive优化的第一性原理——一个字符的LIKE和BETWEEN之差，查询时间差了100倍
2. SELECT * 在Hive上是奢侈的——列式存储下每多读一列都是纯浪费
3. 不要用跑车（MySQL）的标准去评价货运火车（Hive）——它们为完全不同的场景而生

### 🔖 课后拓展

1. **练习**：在你日常使用的Hive/Spark SQL平台上，用EXPLAIN分析你最常用的3条SQL，确认分区裁剪和MAPJOIN是否生效

2. **思考**：如果需要设计一个"数据查询平台"让业务人员也能自助查数，底层应该用Hive还是Presto？为什么？

3. **预习**：明天要学的数据倾斜是Hive性能优化的"终极关卡"——大多数跑不动的SQL，根因是倾斜

4. **面试准备**：背诵以下面试题答案：
   - "Hive和MySQL的核心区别是什么？"
   - "分区是什么？为什么LIKE不触发分区裁剪？"
   - "什么是MAPJOIN？什么时候用它？"
   - "如何优化一个在100亿行数据上跑30分钟的SQL？"

---

*03analyst_thinking · Week 4 Day 4 · Hive与大数据处理*